# Main Figure 5 - SSc pleura segment-free transcript/cell StructureMap

            This notebook is a cleaned, English-commented manuscript code file.

            **Provenance.** `sfplot-manuscript/suppl. fig. 8 new.ipynb`, `sfplot-manuscript/fig. 3 new.ipynb`, and A100 `/data/taobo.hu/projects/ILD` outputs.

            The notebook keeps the original manuscript data paths where they were found. If a path points to
            `/data/taobo.hu` or `/mnt/taobo.hu`, run the notebook on the A100 server or adapt the path to the
            matching mounted Y-drive location.


In [ ]:
# Panel mapping:
# - Figure 5a-e, shared setup. Imports plotting and data libraries used by all SSc pleura segment-free panels.
#

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
sns.set_theme(context="talk", style="white")


In [ ]:
# Panel mapping:
# - Figure 5a-e, raw-data helper. Loads 10x MEX, analysis.zarr.zip, cells.zarr.zip, and transcripts.zarr.zip so cell, landmark, and transcript point tables can be rebuilt.
#

from scipy import sparse
from scipy.sparse import csr_matrix
import scanpy as sc
import fsspec
import zarr
from sfplot import compute_cophenetic_distances_from_df, plot_cophenetic_heatmap


def load_adata_from_mex(mex_dir):
    """Load the 10x MEX matrix used by the SSc pleura analysis."""
    adata = sc.read_10x_mtx(str(mex_dir), var_names="gene_symbols", cache=True)
    adata.var_names_make_unique()
    return adata


def load_cell_groups(analysis_zip, n_cells=None):
    """Read analysis.zarr.zip /cell_groups into sparse cluster membership matrices."""
    uri = Path(analysis_zip).resolve().as_uri()
    store = fsspec.get_mapper(f"zip::{uri}")
    cell_groups = zarr.open_group(store, mode="r")["cell_groups"]
    out = {}
    for key in sorted(cell_groups.group_keys(), key=lambda s: int(s)):
        idx = cell_groups[key]["indices"][:].astype(np.int64)
        indptr = cell_groups[key]["indptr"][:].astype(np.int64)
        n_rows = len(indptr) - 1
        n_cols = (int(idx.max()) + 1) if (n_cells is None and idx.size) else (n_cells or 0)
        data = np.ones_like(idx, dtype=np.uint8)
        out[int(key)] = csr_matrix((data, idx, indptr), shape=(n_rows, n_cols))
    return out


def attach_clusters_from_analysis(adata, analysis_zip):
    """Attach 10x graph-clustering labels as obs['cluster']."""
    membership = load_cell_groups(analysis_zip, n_cells=adata.n_obs)[0]
    nz_per_cell = np.asarray(membership.sum(axis=0)).ravel()
    labels = np.array(membership.argmax(axis=0)).ravel()
    labels[nz_per_cell == 0] = -1
    labels = np.where(labels == -1, -1, labels + 1)
    adata.obs["cluster"] = pd.Categorical(
        np.where(labels == -1, "Unassigned", ["Cluster " + str(x) for x in labels])
    )


def attach_total_counts(adata):
    """Store per-gene total counts in adata.var['total_counts'] for QC."""
    matrix = adata.layers["counts"] if "counts" in adata.layers else adata.X
    feature_types = adata.var.get("feature_types", pd.Series(index=adata.var_names))
    mask = feature_types.eq("Gene Expression").to_numpy()
    cols = np.flatnonzero(mask) if mask.any() else np.arange(adata.n_vars)
    totals = np.asarray(matrix[:, cols].sum(axis=0)).ravel() if sparse.issparse(matrix) else matrix[:, cols].sum(axis=0)
    adata.var.loc[adata.var_names[cols], "total_counts"] = totals


def xenium_cell_id_to_string(prefix, suffix):
    """Convert the two integer arrays in cells.zarr.zip to Xenium cell_id strings."""
    hex_to_ap = str.maketrans("0123456789abcdef", "abcdefghijklmnop")
    return f"{int(prefix):08x}".translate(hex_to_ap) + "-" + str(int(suffix))


def attach_centroids_from_cells(adata, cells_zip):
    """Attach cell centroid coordinates from cells.zarr.zip to adata.obs."""
    root = zarr.group(store=zarr.ZipStore(Path(cells_zip), mode="r"))
    summary = np.asarray(root["cell_summary"][:])
    attrs = dict(root["cell_summary"].attrs.items())
    columns = attrs.get("columns") or attrs.get("col_names") or [
        "cell_centroid_x", "cell_centroid_y", "cell_area",
        "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_area",
        "z_level", "nucleus_count",
    ][: summary.shape[1]]
    cell_df = pd.DataFrame(summary, columns=columns)
    cell_id_parts = np.asarray(root["cell_id"][:])
    cell_df.insert(
        0,
        "cell_id",
        [xenium_cell_id_to_string(prefix, suffix) for prefix, suffix in cell_id_parts],
    )
    cell_df = cell_df.set_index("cell_id").reindex(adata.obs_names)
    adata.obs["x_um"] = cell_df["cell_centroid_x"].to_numpy()
    adata.obs["y_um"] = cell_df["cell_centroid_y"].to_numpy()


def prepare_obs_points(adata, cluster_col="cluster"):
    """Build the x/y/celltype table consumed by Cell-GPS StructureMap."""
    return (
        adata.obs[["x_um", "y_um", cluster_col]]
        .rename(columns={"x_um": "x", "y_um": "y", cluster_col: "celltype"})
        .dropna()
        .reset_index(drop=False)
        .rename(columns={"index": "cell_id"})
    )


def add_landmark_rows(obs_points, adata, landmark_csv):
    """Duplicate pleural landmark cells as a Landmark pseudo-celltype."""
    landmark = pd.read_csv(landmark_csv, comment="#")
    overlap = adata.obs_names.intersection(landmark["Cell ID"].astype(str))
    positions = adata.obs.index.get_indexer(overlap)
    extra = obs_points.iloc[positions].copy()
    extra["celltype"] = "Landmark"
    return pd.concat([obs_points, extra], ignore_index=True)


def collect_gene_xy_um_for_one_gene(root, gid, qmin=20, level=0):
    """Collect x/y coordinates for one gene from transcripts.zarr.zip."""
    grid_level = root["grids"][level]
    chunks = []
    for _, tile in grid_level.groups():
        if "gene_identity" not in tile or "location" not in tile:
            continue
        gene_identity = np.asarray(tile["gene_identity"][:]).ravel()
        location = np.asarray(tile["location"][:])
        if location.ndim != 2:
            location = location.reshape(location.shape[0], -1)
        if gene_identity.shape[0] in location.shape:
            location = location if location.shape[0] == gene_identity.shape[0] else location.T
        else:
            location = location.T if location.shape[0] in (2, 3) else location
        xy = location[:, :2]
        mask = gene_identity == gid
        if qmin is not None:
            if "quality_score" not in tile:
                continue
            mask &= np.asarray(tile["quality_score"][:]).ravel() >= qmin
        if np.any(mask):
            chunks.append(xy[mask])
    return np.vstack(chunks) if chunks else np.empty((0, 2), dtype=float)


def extract_transcript_points(transcripts_zip, genes=None, qv_threshold=20, grids_level=0):
    """Stream transcript x/y coordinates for selected genes from transcripts.zarr.zip."""
    root = zarr.group(store=zarr.ZipStore(Path(transcripts_zip), mode="r"))
    panel = list(map(str, root.attrs.get("gene_names", [])))
    name_to_id = {gene: i for i, gene in enumerate(panel)}
    if genes is None:
        genes = [g for g in panel if g != "Unassigned" and "NegControl" not in g]
    rows = []
    for gene in genes:
        if gene not in name_to_id:
            continue
        xy = collect_gene_xy_um_for_one_gene(root, name_to_id[gene], qmin=qv_threshold, level=grids_level)
        if xy.size:
            gene_df = pd.DataFrame(xy, columns=["x", "y"])
            gene_df.insert(0, "celltype", gene)
            rows.append(gene_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["celltype", "x", "y"])


In [ ]:
# Panel mapping:
# - Figure 5a. Reads the manuscript segment-free StructureMap table and draws the circular dendrogram panel for SSc pleura.
#

from sfplot.plotting.circular_dendrogram import plot_circular_dendrogram_pycirclize

RAW_DIR = Path("/data/taobo.hu/ILD/SSc_2_1_1_raw")
LANDMARK_DIR = Path("D:/GitHub/sfplot/sfplot-manuscript/t_by_c_SSc_2_1_1_Landmark")
LANDMARK_CSV = Path("/data/taobo.hu/sfplot-manuscript/pleural_cells_stats.csv")
if not LANDMARK_CSV.exists():
    LANDMARK_CSV = Path("D:/GitHub/sfplot/sfplot-manuscript/pleural_cells_stats.csv")
OUTPUT_DIR = Path("outputs/main_figure_5_SSc_pleura_segment_free")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the manuscript StructureMap output table to draw the circular dendrogram panel.
tbc_result = pd.read_csv(LANDMARK_DIR / "t_and_c_result_SSc_2_1_1.csv", index_col=0)
segment_free = pd.read_csv(LANDMARK_DIR / "StructureMap_table_SSc_2_1_1_Landmark_Segment_free.csv", index_col=0)
plot_circular_dendrogram_pycirclize(segment_free, output_pdf=str(OUTPUT_DIR / "SSc_pleura_circular_dendrogram.pdf"), metric="euclidean", method="average", leaf_label_size=7, figsize=(13, 13))


In [ ]:
# Panel mapping:
# - Figure 5b-c. Recreates the segment-free cell/Landmark point table from raw 10x/zarr inputs and recomputes the Cell-GPS StructureMap.
#

# Recreate the segment-free cell/Landmark point table from the raw 10x MEX/zarr files.
adata = load_adata_from_mex(RAW_DIR / "cell_feature_matrix")
attach_clusters_from_analysis(adata, RAW_DIR / "analysis.zarr.zip")
attach_total_counts(adata)
attach_centroids_from_cells(adata, RAW_DIR / "cells.zarr.zip")

valid = (~adata.obs["cluster"].isna()) & (adata.obs["cluster"] != "Unassigned")
adata = adata[valid].copy()
points = prepare_obs_points(adata)
if LANDMARK_CSV.exists():
    points = add_landmark_rows(points, adata, LANDMARK_CSV)

POINTS_CSV = OUTPUT_DIR / "SSc_2_1_1_segment_free_points.csv"
points.to_csv(POINTS_CSV, index=False)

row_coph, _ = compute_cophenetic_distances_from_df(points, x_col="x", y_col="y", celltype_col="celltype", chunk_rows=2_000_000)
row_coph.to_csv(OUTPUT_DIR / "StructureMap_table_SSc_2_1_1_Landmark_Segment_free_recomputed.csv")
plot_cophenetic_heatmap(row_coph, matrix_name="row_coph", output_filename=str(OUTPUT_DIR / "StructureMap_of_SSc_2_1_1_Landmark_Segment_free_recomputed.pdf"), sample="SSc_2_1_1_Landmark_Segment_free")


In [ ]:
# Panel mapping:
# - Figure 5d-e. Streams selected transcript coordinates from transcripts.zarr.zip and draws the SSc gene spatial panels over the cell background.
#

# Plot the manuscript gene spatial panels by streaming transcript coordinates from transcripts.zarr.zip.
genes = ["CCL4", "IL10", "CCL3", "MFAP5", "CXCL8", "LYVE1", "IGF1", "WT1", "IFNG", "UPK1B"]
transcripts = extract_transcript_points(RAW_DIR / "transcripts.zarr.zip", genes=genes, qv_threshold=20, grids_level=0)
transcripts.to_csv(OUTPUT_DIR / "SSc_2_1_1_selected_transcript_points.csv", index=False)

for gene in genes:
    sub = transcripts[transcripts["celltype"].eq(gene)]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(points["x"], points["y"], s=0.03, c="lightgrey", rasterized=True)
    ax.scatter(sub["x"], sub["y"], s=0.6, c="#D55E00", rasterized=True)
    ax.set_title(gene)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    fig.savefig(OUTPUT_DIR / f"SSc_gene_{gene}.pdf", bbox_inches="tight")
    plt.close(fig)
